In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
import json
from datasets import Dataset

In [2]:
with open("train_spider.json", "r") as f:
    train_data = json.load(f)
with open("tables.json", "r") as f:
    table_data = json.load(f)

db_map = {}
for db in table_data:
    schema = []
    for i, tab_name in enumerate(db['table_names_original']):
        cols = [c[1] for c in db['column_names_original'] if c[0] == i]
        schema.append(f"{tab_name}({', '.join(cols)})")
    db_map[db['db_id']] = " | ".join(schema)

prompt_template = """### Instruction:
Convert the natural language question into a SQL query using the provided schema.

### Input:
Schema: {}
Question: {}

### Response:
{}"""

formatted_data = []
for entry in train_data:
    full_text = prompt_template.format(
        db_map.get(entry['db_id'], "Unknown"),
        entry['question'],
        entry['query']
    )
    formatted_data.append({"text": full_text})

dataset = Dataset.from_list(formatted_data)
print(f"Dataset prepared: {len(dataset)} samples ready for training.")

Dataset prepared: 7000 samples ready for training.


In [3]:
max_seq_length = 2048 
dtype = None
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Model ready on RTX 3070 Ti.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3070 Ti. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.3.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model ready on RTX 3070 Ti.


In [4]:
OUTPUT_DIR = "sql_model_lora"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Training complete. Model saved to: {OUTPUT_DIR}")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/7000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 2 | Total steps = 1,750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,768,832 of 3,880,848,384 (1.54% trained)


Step,Training Loss
1,1.477940
2,1.588328
3,1.376709
4,1.311393
5,1.423555
6,1.441688
7,1.177417
8,1.277749
9,1.081126
10,0.975012


Training complete. Model saved to: sql_model_lora
